# 🌍 Spatial-LLM — Per-module gating, NO coordinate leak (Kaggle)

**Why this experiment exists:** the first run (coords in the prompt text) hit
balanced acc ~0.99 for *both* shared and per-module gating — but every fusion gate
stayed at ~0. The model was just reading lat/lon off the text; the spatial pathway
was unused, so there was nothing for gating to differentiate.

**The fix:** `coords_in_text=false`. Location is removed from the text and reaches the
model ONLY through the spatial channel (coord/elevation embedder + grid cells). Every
elevation example then has identical text, so the gates MUST open to solve the task.

**Arms (coords-free data, 3D coords via channel — only the gating differs):**
- `shared` → `configs/coord_3d_noleak.yaml`        (one gate for all modules)
- `permod` → `configs/coord_3d_permod_noleak.yaml` (one gate per module)

**Read it as:** balanced acc near 0.5 = the pathway can't learn this even when forced;
well above 0.5 = the spatial stack works, and the gate read-out shows which module
carries it. Then shared-vs-permod is finally a real comparison.

Setup: GPU T4 x2 (or P100) + Internet ON. Start with `SEEDS=[42]`; for the full sweep
use Save Version → Save & Run All (background execution).

## 1. Setup — clone/pull `main`, install deps, pre-download model

In [ ]:
%cd /kaggle/working
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'        # Xet streaming backend stalls on Kaggle
os.environ['CUDA_VISIBLE_DEVICES'] = '0'      # single GPU (model isn't DataParallel-safe)
if not os.path.isdir('Spatial-LLM'):
    !git clone https://github.com/Mohammadzamanid/Spatial-LLM.git
%cd /kaggle/working/Spatial-LLM
!git pull origin main          # pull the coords_in_text fix + no-leak configs
!pip install -q transformers peft accelerate datasets bitsandbytes
!pip install -q geopandas shapely mercantile pyyaml pillow
import torch
print('commit below (want the coords_in_text fix):')
!git log --oneline -1
print('GPUs visible to notebook:', torch.cuda.device_count())
from huggingface_hub import snapshot_download
print('model cached at:', snapshot_download('Qwen/Qwen2.5-1.5B'))

## 2. Data — elevation task, NO coords in text
`--no-coords-in-text` strips lat/lon from the question. The print should show a
question with NO numbers in it — that's the whole point.

In [ ]:
!python -m src.data.real_datasets --dataset cities15000 --task elevation --no-coords-in-text --n_train 8000 --n_val 1000
import json
r = json.loads(open('data/processed/val.jsonl').readline())
print('Q:', r['question'])          # should contain NO coordinates
print('A:', r['answer'], '| (elevation', r.get('elevation'), 'reaches the model only via coords tensor)')

## 3. Config — arms & seeds (shared by Train/Eval cells)

In [ ]:
ARMS = {'shared': 'configs/coord_3d_noleak.yaml',          # one shared gate
        'permod': 'configs/coord_3d_permod_noleak.yaml'}    # one gate per module
SEEDS = [42]          # validate first; then set [42, 43, 44] and re-run Train + Eval
print('arms:', list(ARMS), '| seeds:', SEEDS)

## 4. Train
~10–20 min/run on a T4. Output streams live (`!` + `python -u`). After the banner
there's a ~30–60 s silent import gap before the first log line — that's normal.

In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
for arm, cfg in ARMS.items():
    for s in SEEDS:
        out = f'outputs/{arm}_seed{s}'
        print(f'\n===== TRAIN {arm} seed={s} -> {out} =====', flush=True)
        !CUDA_VISIBLE_DEVICES=0 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True TOKENIZERS_PARALLELISM=false python -u -m src.training.trainer --config {cfg} --seed {s} --output_dir {out}

## 5. Evaluate — balanced accuracy + the gate read-out
Now the gates matter. If the spatial pathway is working you should see |gate| clearly
above ~0 (unlike the ~0.001 from the leaky run), and balanced acc well above 0.5.

In [ ]:
import subprocess, os, json, re, statistics as st
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0', 'TOKENIZERS_PARALLELISM': 'false'}
results = []
for arm, cfg in ARMS.items():
    for s in SEEDS:
        out, label = f'outputs/{arm}_seed{s}', f'{arm}_seed{s}'
        r = subprocess.run(['python','-m','src.eval.accuracy','--config',cfg,
                            '--checkpoint',out,'--val','data/processed/val.jsonl',
                            '--dump-gates','--seed',str(s),'--label',label],
                           capture_output=True, text=True, env=env)
        print(r.stdout[-1800:])
        m = re.search(r'===RESULT-JSON-START===\n(.*)\n===RESULT-JSON-END===', r.stdout, re.S)
        if m: results.append(json.loads(m.group(1)))
        else: print('⚠️ no result block for', label, '\n', r.stderr[-800:])

print('\n================ SUMMARY (balanced accuracy) ================')
by_arm = {}
for x in results:
    by_arm.setdefault(x['label'].split('_seed')[0], []).append(x['balanced_accuracy'])
for arm, accs in by_arm.items():
    accs = [a for a in accs if a is not None]
    mean = st.mean(accs); sd = st.pstdev(accs) if len(accs) > 1 else 0.0
    print(f'  {arm:8s} {mean:.3f} ± {sd:.3f}  (n={len(accs)}: {[round(a,3) for a in accs]})')

## 6. Copy this block back into the chat
Paste the `===PASTE-THIS-BACK===` block; it gets committed to
`results/per_module_gating_noleak.json` on `main`.

In [ ]:
import json, os
payload = {'experiment':'per_module_gating_noleak','task':'elevation',
           'dataset':'cities15000','coords_in_text':False,'metric':'balanced_accuracy',
           'source':'kaggle','reproduced_in_repo':True,'seeds':SEEDS,'runs':results}
os.makedirs('/kaggle/working/results', exist_ok=True)
with open('/kaggle/working/results/per_module_gating_noleak.json','w') as f:
    json.dump(payload, f, indent=2)
print('saved -> /kaggle/working/results/per_module_gating_noleak.json')
print('\n===PASTE-THIS-BACK===')
print(json.dumps(payload, indent=2))
print('===END===')